In [4]:
# 环境
import sys
from pathlib import Path

# # 从当前目录向上找 .git，定位根目录
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT/".git").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
# # 把项目根目录加入到sys.path，
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))
# 导入路径配置
from src.config import (
    CLEAN_PARQUET,
    DB_PATH,
    PROJECT_ROOT,
    RAW_CSV,
    RAW_PARQUET,
    ensure_dirs,
)

ensure_dirs()
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_CSV:", RAW_CSV)
print("RAW_PARQUET:", RAW_PARQUET)
print("CLEAN_PARQUET:", CLEAN_PARQUET)
print("DB_PATH:", DB_PATH)


PROJECT_ROOT: d:\数据分析\Project\ecommerce_analysis
RAW_CSV: d:\数据分析\Project\ecommerce_analysis\data\raw\user_behavior.csv
RAW_PARQUET: d:\数据分析\Project\ecommerce_analysis\data\parquet\user_behavior_raw.parquet
CLEAN_PARQUET: d:\数据分析\Project\ecommerce_analysis\data\processed\user_behavior_clean.parquet
DB_PATH: d:\数据分析\Project\ecommerce_analysis\data\database\user_behavior.duckdb


In [5]:
import duckdb

con = duckdb.connect(str(DB_PATH),read_only=True)
# 查看前五条样本
con.sql("SELECT * FROM user_behavior LIMIT 5").df()


,user_id,item_id,behavior_type,item_category,time
0,98047837,232431562,浏览,4245,2014-12-06 02:00:00
1,97726136,383583590,浏览,5894,2014-12-09 20:00:00
2,98607707,64749712,浏览,2883,2014-12-18 11:00:00
3,98662432,320593836,浏览,6562,2014-12-06 10:00:00
4,98145908,290208520,浏览,13926,2014-12-16 21:00:00


In [6]:
# 数据概览
con.sql("""
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_id) AS users,
    COUNT(DISTINCT item_id) AS items,
    COUNT(DISTINCT item_category) AS categories,
    MIN(time) AS min_time,
    MAX(time) AS max_time,
    DATE_DIFF('day',MIN(time), MAX(time)) AS days
FROM user_behavior
""").df()

,total_rows,users,items,categories,min_time,max_time,days
0,6213379,10000,2876947,8916,2014-11-18,2014-12-18 23:00:00,30


In [7]:
# 查看行为分类，各占多少？
con.sql("""
SELECT behavior_type,
    COUNT(*) AS cnt,
    COUNT(DISTINCT user_id) AS uv,
    ROUND(COUNT(*) *100 / SUM(COUNT(*)) OVER (),2) AS pct
FROM user_behavior
GROUP BY behavior_type
ORDER BY pct DESC
""").df()

,behavior_type,cnt,uv,pct
0,浏览,5535879,10000,89.10
1,加购,331350,8614,5.33
2,收藏,239472,6730,3.85
3,购买,106678,8886,1.72


In [8]:
# 表的空值情况
con.sql("DESCRIBE user_behavior").df()
con.sql("""
SELECT 
    COUNT(*) AS total,
    SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS null_user,
    SUM(CASE WHEN item_id IS NULL THEN 1 ELSE 0 END) AS null_item,
    SUM(CASE WHEN behavior_type IS NULL THEN 1 ELSE 0 END) AS null_behavior_type,
    SUM(CASE WHEN item_category IS NULL THEN 1 ELSE 0 END) AS null_item_category,
    SUM(CASE WHEN time IS NULL THEN 1 ELSE 0 END) AS null_time
FROM user_behavior
""").df()

,total,null_user,null_item,null_behavior_type,null_item_category,null_time
0,6213379,0.0,0.0,0.0,0.0,0.0


In [9]:
# behavior_type异常值检查
con.sql("""
SELECT behavior_type, COUNT(*) AS cnt
FROM user_behavior
WHERE behavior_type IS NULL OR behavior_type NOT IN ('浏览','加购','收藏','购买')
GROUP BY behavior_type
""").df()

,behavior_type,cnt


In [10]:
# 用户行为异常检查
con.sql("""
SELECT 
    MIN(cnt) AS min_actions,
    MAX(cnt) AS max_actions,
    AVG(cnt) AS avg_actions,
    MEDIAN(cnt) AS median_actions
FROM (
    SELECT user_id, COUNT(*) as cnt
    FROM user_behavior
    GROUP BY user_id
)
""").df()

,min_actions,max_actions,avg_actions,median_actions
0,1,15936,621.3379,372.0


In [11]:
# 商品热度异常检查
con.sql("""
SELECT 
    MIN(cnt) AS min_pv,
    MAX(cnt) AS max_pv,
    AVG(cnt) AS avg_pv,
    MEDIAN(cnt) AS median_pv
FROM (
SELECT item_id, COUNT(*) AS cnt,
FROM user_behavior
GROUP BY item_id
)
""").df()

,min_pv,max_pv,avg_pv,median_pv
0,1,730,2.159713,1.0


In [12]:
# 时间异常检查
con.sql("""
SELECT 
    MIN(time) AS min_time,
    MAX(time) AS max_time,
    SUM(IF(time < '2014-11-18',1,0)) AS ancient_time,
     SUM(IF(time > '2014-12-19',1,0)) AS future_time
FROM user_behavior
""").df()

,min_time,max_time,ancient_time,future_time
0,2014-11-18,2014-12-18 23:00:00,0.0,0.0


In [13]:
# 重复值检查
con.sql("""
SELECT COUNT(*) AS duplicates_num
FROM(
    SELECT *, COUNT(*) AS cnt
    FROM user_behavior
    GROUP BY user_id, item_id,behavior_type,item_category, time
    HAVING COUNT(*) > 1
)

""").df()

,duplicates_num
0,0


In [14]:
# 时间维度检查 每日用户访问量，商品访问量
con.sql("""
SELECT 
    MAX(uv) AS max_uv,
    MIN(uv) AS min_uv,
    MEDIAN(uv) AS median_uv,
    AVG(uv) AS avg_uv,
    MAX(items) AS max_items,
    MIN(items) AS min_items,
    MEDIAN(items) AS median_items,
    AVG(items) AS avg_items,
    MAX(actions) AS max_actions,
    MIN(actions) AS min_actions,
    MEDIAN(actions) AS median_actions,
    AVG(actions) AS avg_actions,
FROM(
    SELECT 
        DATE_TRUNC('day',time) AS dt,
        COUNT(DISTINCT user_id) AS uv,
        COUNT(DISTINCT item_id) AS items,
        COUNT(*) AS actions
    FROM user_behavior
    GROUP BY dt 
    ORDER BY dt
)

""").df()

,max_uv,min_uv,median_uv,avg_uv,max_items,min_items,median_items,avg_items,max_actions,min_actions,median_actions,avg_actions
0,7720,6187,6513.0,6526.580645,232398,129629,149263.0,151083.354839,346464,168903,197172.0,200431.580645


In [15]:
# 时间维度检查 时段分布
con.sql("""
SELECT 
    EXTRACT(hour FROM time) AS hour,
    COUNT(*) AS actions,
    COUNT(DISTINCT item_id) AS pv, 
    COUNT(DISTINCT user_id) AS uv
FROM user_behavior
GROUP BY hour
ORDER BY hour
""").df()

,hour,actions,pv,uv
0,0,264993,213789,5786
1,1,138190,117612,3780
2,2,76170,66416,2532
3,3,51872,45830,1937
4,4,41865,37021,1765
5,5,45623,39806,2030
6,6,81224,68286,3564
7,7,146054,119928,5722
8,8,199962,160911,7108
9,9,244014,193458,7734


In [16]:
# 时间维度检查 星期分布
con.sql("""
SELECT 
    CASE DAYOFWEEK(time)
        WHEN 0 THEN '星期日'
        WHEN 1 THEN '星期一'
        WHEN 2 THEN '星期二'
        WHEN 3 THEN '星期三'
        WHEN 4 THEN '星期四'
        WHEN 5 THEN '星期五'
        WHEN 6 THEN '星期六'
        ELSE '未知'
    END AS week,
    COUNT(*) AS actions,
    COUNT(DISTINCT item_id) AS pv,
    COUNT(DISTINCT user_id) AS uv
FROM user_behavior
GROUP BY week
ORDER BY MIN(DAYOFWEEK(time))
""").df()

,week,actions,pv,uv
0,星期日,804865,553707,9193
1,星期一,790073,539984,9281
2,星期二,981404,656152,9428
3,星期三,983358,657307,9465
4,星期四,1008972,667402,9474
5,星期五,871392,569380,9312
6,星期六,773315,533102,9194


In [17]:
# 用户维度 
# 用户行为数分布,分为只来1次、2-5次、6-20次、21-100次
con.sql("""
with user_stats AS(
    SELECT user_id, COUNT(*) AS actions
    FROM user_behavior
    GROUP BY user_id
)
SELECT 
    CASE
        WHEN actions = 1 THEN '1次'
        WHEN actions >= 2 and actions <=100 THEN '2-100次'
        WHEN actions >= 101 and actions <=200 THEN '101-200次'
        WHEN actions >= 201 and actions <=300 THEN '201-300次'
        WHEN actions >= 301 and actions <=400 THEN '301-400次'
        ELSE '400次以上'
    END AS action_level,
    COUNT(*) AS user_cnt
FROM user_stats
GROUP BY action_level
ORDER BY MIN(actions)

""").df()


,action_level,user_cnt
0,1次,8
1,2-100次,1575
2,101-200次,1532
3,201-300次,1192
4,301-400次,939
5,400次以上,4754


In [18]:
# 用户维度 购买用户占比
con.sql("""
SELECT 
    COUNT(DISTINCT user_id) AS total_user,
    COUNT(DISTINCT CASE WHEN behavior_type = '购买' THEN user_id END) AS buyer_cnt,
    ROUND(
        COUNT(DISTINCT CASE WHEN behavior_type = '购买' THEN user_id END) * 100 / COUNT(DISTINCT user_id),2
    ) AS buyer_rate
FROM user_behavior
""").df()

,total_user,buyer_cnt,buyer_rate
0,10000,8886,88.86


In [19]:
# 品类分布
con.sql("""
SELECT 
    item_category,
    COUNT(*) AS pv,
    COUNT(DISTINCT user_id) as users,
    COUNT(DISTINCT item_id) as items,
    SUM(IF(behavior_type == '购买',1,0)) AS buy_actions,
    ROUND(SUM(IF(behavior_type == '购买',1,0))*100 / COUNT(*),2) AS buy_cvr
FROM user_behavior
GROUP BY item_category
ORDER BY pv DESC
LIMIT 20
""").df()

,item_category,pv,users,items,buy_actions,buy_cvr
0,1863,201110,5997,83354,1910.0,0.95
1,13230,177948,5619,74067,825.0,0.46
2,5894,169451,5279,83758,927.0,0.55
3,5027,168103,5499,60323,815.0,0.48
4,6513,152148,5288,65045,1004.0,0.66
5,5399,141726,5312,54559,1014.0,0.72
6,11279,94686,4715,47538,703.0,0.74
7,2825,84911,4551,43997,578.0,0.68
8,5232,71772,4972,25900,1384.0,1.93
9,10894,71129,4523,33482,811.0,1.14


In [20]:
# 商品热度 Top20
con.sql("""
SELECT 
    item_id,
    COUNT(*) AS pv,
    SUM(IF(behavior_type == '浏览',1,0)) AS visit_cnt,
    SUM(IF(behavior_type == '收藏',1,0)) AS save_cnt,
    SUM(IF(behavior_type == '加购',1,0)) AS addcart_cnt,
    SUM(IF(behavior_type == '购买',1,0)) AS buy_cnt,
    ROUND(SUM(IF(behavior_type == '购买',1,0)) * 100 / COUNT(*),2) AS buy_rate
FROM user_behavior
GROUP BY item_id
ORDER BY pv DESC
LIMIT 20
""").df()

,item_id,pv,visit_cnt,save_cnt,addcart_cnt,buy_cnt,buy_rate
0,112921337,730,716.0,5.0,8.0,1.0,0.14
1,97655171,589,550.0,7.0,19.0,13.0,2.21
2,387911330,519,473.0,10.0,17.0,19.0,3.66
3,135104537,493,478.0,7.0,8.0,0.0,0.00
4,2217535,425,412.0,1.0,11.0,1.0,0.24
5,128186279,423,395.0,13.0,14.0,1.0,0.24
6,5685392,403,377.0,12.0,12.0,2.0,0.50
7,277922302,401,356.0,24.0,20.0,1.0,0.25
8,275450912,389,382.0,5.0,2.0,0.0,0.00
9,14087919,375,300.0,15.0,33.0,27.0,7.20


# 探查结论

- 数据量：6213379 行，10000 个用户，2876947 个商品，8916 个品类
- 时间范围：2014-11-18 ~ 2014-12-18，共 30 天
- 行为分布：浏览：89.1%，加购：5.33%，收藏：3.85%，购买：1.72%
- 缺失情况：没有字段缺失
- 重复记录：0 组
- 异常值：无
- 用户分布：0.08%的用户只活跃 1 次
- 购买用户占比：88.86%的用户有股买行为
- 清洗建议：非常干净的数据集（因为我已经清洗过了）